# 패키지 설치하기

TPU 관련 패키지를 설치합니다. TPU 사용시 아래 라인 첫 문자(#)를 지우고 수행하세요. GPU를 쓴다면 아래 라인을 실행할 필요가 없습니다.

In [ ]:
# TPU 사용시 아래 라인 첫 문자(#)를 지우고 수행하세요.
#!pip install cloud-tpu-client==0.10 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

**구버전 pip로 다운그레이드 하기**<br>
최신 pip는 구버전 라이브러리의 와일드카드 메타데이터(>=1.8.*)를 거부하므로, 규격 검증이 유연한 24.1 미만 버전으로 먼저 낮춥니다.


In [4]:
!pip install "pip<24.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2


**Cython 버전 맞추기**<br>
구버전 PyYAML을 소스에서 빌드할 때 발생하는 컴파일 오류를 막기 위해 호환되는 구버전 cython을 설치합니다.

In [5]:
!pip install "cython<3.0.0"

TPU 이외에 의존성 있는 패키지를 설치합니다.

# 구글 드라이브 연동하기
모델 체크포인트 등을 저장해 둘 구글 드라이브를 연결합니다. 자신의 구글 계정에 적용됩니다.

In [1]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

ModuleNotFoundError: No module named 'google.colab'

# 각종 설정
모델 하이퍼파라메터(hyperparameter)와 저장 위치 등 설정 정보를 선언합니다.

In [1]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA를 사용할 수 없습니다. CUDA 지원 PyTorch가 설치된 Python 환경을 선택하고 "
        "노트북 커널을 재시작한 뒤 다시 실행하세요."
    )

print(f"학습 GPU: {torch.cuda.get_device_name(0)}")

# 1. 모델명 및 경로 설정
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./checkpoint-doccls"
MAX_SEQ_LENGTH = 128

# 2. 토크나이저 및 모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 3. NSMC 데이터셋 로드 (허깅페이스 데이터셋 스크립트 차단 우회를 위해 원본 TSV 주소에서 직접 로드)
dataset = load_dataset(
    "csv",
    data_files={
        "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
    },
    delimiter="\t"
)

# 결측치(Null)가 있는 행 제거 및 document 컬럼을 string 타입으로 변환
dataset = dataset.filter(lambda x: x['document'] is not None)

# 4. 데이터 전처리(토크나이제이션) 함수 정의
def tokenize_function(examples):
    return tokenizer(
        examples["document"],
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        truncation=True
    )

# 데이터셋에 토큰화 적용
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. TrainingArguments 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=3,
    seed=7,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=500,
)

# 6. Trainer 초기화 및 학습 실행 (최신 버전에 맞추어 tokenizer 대신 processing_class를 사용합니다)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
)

# 학습 시작
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

RuntimeError: CUDA를 사용할 수 없습니다. CUDA 지원 PyTorch가 설치된 Python 환경을 선택하고 노트북 커널을 재시작한 뒤 다시 실행하세요.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "./checkpoint-doccls"
MAX_SEQ_LENGTH = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = probabilities.argmax().item()
    labels = {0: "부정", 1: "긍정"}

    return {
        "sentiment": labels[predicted_id],
        "confidence": probabilities[predicted_id].item(),
        "negative_probability": probabilities[0].item(),
        "positive_probability": probabilities[1].item(),
    }

result = predict_sentiment("기대 안 했는데 정말 재미있었어요!")
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'sentiment': '긍정', 'confidence': 0.9930096864700317, 'negative_probability': 0.006990361027419567, 'positive_probability': 0.9930096864700317}


In [17]:
import os

# 설정한 출력 경로 확인
if os.path.exists(OUTPUT_DIR):
    print(f"[정상] '{OUTPUT_DIR}' 경로가 존재합니다.")
    print("\n--- 생성된 산출물 파일 목록 ---")
    for root, dirs, files in os.walk(OUTPUT_DIR):
        level = root.replace(OUTPUT_DIR, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files[:10]:  # 최대 10개만 출력
            print(f"{subindent}{f}")
else:
    print(f"[주의] '{OUTPUT_DIR}' 경로를 찾을 수 없습니다. 학습이 아직 완료되지 않았거나 경로 설정을 확인해 주세요.")

[주의] '/gdrive/My Drive/nlpbook/checkpoint-doccls' 경로를 찾을 수 없습니다. 학습이 아직 완료되지 않았거나 경로 설정을 확인해 주세요.


In [6]:
!pip install ratsnlp

  Using cached ratsnlp-1.0.53-py3-none-any.whl.metadata (741 bytes)
  Using cached pytorch_lightning-1.6.1-py3-none-any.whl.metadata (33 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 2.0 MB/s eta 0:00:00
  Using cached Korpora-0.2.0-py3-none-any.whl.metadata (26 kB)
  Using cached flask_ngrok-0.0.25-py3-none-any.whl.metadata (1.8 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 582.5/582.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4

In [ ]:
import torch
from ratsnlp.nlpbook.classification import ClassificationTrainArguments
args = ClassificationTrainArguments(
    pretrained_model_name="beomi/kcbert-base",
    downstream_corpus_name="nsmc",
    downstream_model_dir="/gdrive/My Drive/nlpbook/checkpoint-doccls",
    batch_size=32 if torch.cuda.is_available() else 4,
    learning_rate=5e-5,
    max_seq_length=128,
    epochs=3,
    tpu_cores=0 if torch.cuda.is_available() else 8,
    seed=7,
)

# 랜덤 시드 고정
학습 재현을 위해 랜덤 시드를 고정합니다.

In [ ]:
from ratsnlp import nlpbook
nlpbook.set_seed(args)

# 로거 설정
메세지 출력 등을 위한 logger를 설정합니다.

In [ ]:
nlpbook.set_logger(args)

# 말뭉치 다운로드
실습에 사용할 말뭉치(Naver Sentiment Movie review Corpus)를 다운로드합니다.

In [ ]:
from Korpora import Korpora
Korpora.fetch(
    corpus_name=args.downstream_corpus_name,
    root_dir=args.downstream_corpus_root_dir,
    force_download=True,
)

# 토크나이저 준비
토큰화를 수행하는 토크나이저를 선언합니다

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained(
    args.pretrained_model_name,
    do_lower_case=False,
)

# 학습데이터 구축
학습데이터를 만듭니다.

In [ ]:
from torch.utils.data import DataLoader, SequentialSampler, RandomSampler
from ratsnlp.nlpbook.classification import NsmcCorpus, ClassificationDataset
corpus = NsmcCorpus()
train_dataset = ClassificationDataset(
    args=args,
    corpus=corpus,
    tokenizer=tokenizer,
    mode="train",
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    sampler=RandomSampler(train_dataset, replacement=False),
    collate_fn=nlpbook.data_collator,
    drop_last=False,
    num_workers=args.cpu_workers,
)

# 테스트 데이터 구축
학습 중에 평가할 테스트 데이터를 구축합니다.

In [ ]:
val_dataset = ClassificationDataset(
    args=args,
    corpus=corpus,
    tokenizer=tokenizer,
    mode="test",
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    sampler=SequentialSampler(val_dataset),
    collate_fn=nlpbook.data_collator,
    drop_last=False,
    num_workers=args.cpu_workers,
)


# 모델 초기화
프리트레인이 완료된 BERT 모델을 읽고, 문서 분류를 수행할 모델을 초기화합니다.

In [ ]:
from transformers import BertConfig, BertForSequenceClassification
pretrained_model_config = BertConfig.from_pretrained(
    args.pretrained_model_name,
    num_labels=corpus.num_labels,
)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
        args.pretrained_model_name,
        config=pretrained_model_config,
)

# 학습 준비
Task와 Trainer를 준비합니다.

In [ ]:
from ratsnlp.nlpbook.classification import ClassificationTask
task = ClassificationTask(model, args)

In [ ]:
trainer = nlpbook.get_trainer(args)

# 학습
준비한 데이터와 모델로 학습을 시작합니다. 학습 결과물(체크포인트)은 미리 연동해둔 구글 드라이브의 준비된 위치(`/gdrive/My Drive/nlpbook/checkpoint-doccls`)에 저장됩니다.

In [ ]:
trainer.fit(
    task,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)